# Practica de Programacion y Algoritmia Avanzada – GIA

El objetivo de esta actividad es poner en practica los conocimientos de programacion dinamica, para tal caso, se os propone implementar el algoritmo determinista Cocke–Kasami–Younger.

El algoritmo CKY determinista es un algoritmo utilizado para analizar la estructura sintactica de cadenas de texto, y determinar si pueden ser derivadas apartir de las reglas de una gramatica.

In [2]:
def cargar_gramatica(ruta_archivo):
    reglas = {}

    with open(ruta_archivo) as archivo:
        for linea in archivo:
            linea = linea.strip()
            if '->' in linea:
                lhs, rhs = linea.split('->')
                lhs = lhs.strip()
                alternativas = rhs.split('|')

                for alt in alternativas:
                    produccion = alt.strip().split()
                    if produccion:
                        reglas.setdefault(lhs, []).append(produccion)

    return reglas


In [3]:
# gramatica= cargar_gramatica('c:/Users/veron/Documents/UNI/2º/Febrero/PAA/Practica/input.txt')

In [ ]:
def es_cnf(gramatica):
    no_terminales = set(gramatica.keys())
    terminales = set()

    # Recorremos toda la gramática para ver qué símbolos podrían ser terminales
    for reglas in gramatica.values():
        for produccion in reglas:
            for simbolo in produccion:
                if simbolo not in no_terminales:
                    terminales.add(simbolo)

    # Ahora comprobamos si cada producción cumple las reglas de la CNF
    for _, reglas in gramatica.items():
        for produccion in reglas:
            if len(produccion) == 1:
                # Caso A -> a
                if produccion[0] not in terminales:
                    return False
            elif len(produccion) == 2:
                # Caso A -> B C
                if produccion[0] in terminales or produccion[1] in terminales:
                    return False
            else:
                # Producción con más de 2 símbolos ej A-> BCD
                return False

    return True


In [5]:
def cky(gramatica, palabra):
    n = len(palabra)
    tabla = [[set() for _ in range(n)] for _ in range(n)]

    # Crear un diccionario inverso: RHS -> LHS
    inversa = {}
    for lhs, producciones in gramatica.items():
        for rhs in producciones:
            clave = tuple(rhs)
            inversa.setdefault(clave, []).append(lhs)

    # Rellenar la diagonal (longitud 1): A -> a
    for i, simbolo in enumerate(palabra):
        if (simbolo,) in inversa:
            tabla[i][i].update(inversa[(simbolo,)])

    # Rellenar el resto de la tabla
    for longitud in range(2, n + 1):  # tamaño del fragmento
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            for k in range(i, j):
                izquierda = tabla[i][k]
                derecha = tabla[k + 1][j]
                for B in izquierda:
                    for C in derecha:
                        if (B, C) in inversa:
                            tabla[i][j].update(inversa[(B, C)])

    # Verificamos si el símbolo inicial S genera toda la cadena
    return 'S' in tabla[0][n - 1]


In [6]:
# Ejemplo de uso
# Usamos las dos gramáticas proporcionadas en el enunciado

Gramar1 = {
    'S': [['a'], ['X', 'A'], ['A', 'X'], ['b']],
    'A': [['R', 'B']],
    'B': [['A', 'X'], ['b'], ['a']],
    'X': [['a']],
    'R': [['X', 'B']]
}

Gramar2 = {
    'S': [['A','B'], ['C','D'], ['C','B'], ['S','S']],
    'A': [['B','C'], ['a']],
    'B': [['S','C'], ['b']],
    'C': [['D','D'], ['b']],
    'D': [['B','A']]
}

print("Grammar in CNF:", es_cnf(Gramar1)) # True
print("Grammar in CNF:", es_cnf(Gramar2)) # True

print("Can generate 'abab':", cky(Gramar1, ['a', 'b', 'a', 'b'])) # False
print("Can generate 'abba':", cky(Gramar1, ['a', 'b', 'b', 'a'])) # True



Grammar in CNF: True
Grammar in CNF: True
Can generate 'abab': False
Can generate 'abba': True


In [7]:
# EXTENSION 1
# Transformacion de cualquier gramatica CFG a CNF.

# Funciones auxiliares para la transformación a CNF

def eliminar_unitarias(gramatica):
    """Elimina las producciones unitarias de la gramática."""
    unitarias = set()
    
    # Encontramos las producciones unitarias
    for A, reglas in gramatica.items():
        for regla in reglas:
            if len(regla) == 1 and regla[0].isupper():
                unitarias.add((A, regla[0]))
    
    # Sustituir las producciones unitarias
    for A, B in unitarias:
        for regla_B in gramatica[B]:
            if regla_B not in gramatica[A]:
                gramatica[A].append(regla_B)
    
    # Eliminar las producciones unitarias
    for A, B in unitarias:
        gramatica[A] = [regla for regla in gramatica[A] if regla != [B]]
    
    return gramatica


def eliminar_vacias(gramatica):
    """Elimina las producciones vacías (A -> ε)."""
    nullable = set()

    # Encontramos las variables que pueden generar ε
    for A, reglas in gramatica.items():
        for regla in reglas:
            if regla == []:
                nullable.add(A)

    # Eliminar las producciones vacías y generar nuevas producciones
    cambios = True
    while cambios:
        cambios = False
        for A in list(gramatica):
            # Eliminar las producciones vacías
            nuevas_reglas = [regla for regla in gramatica[A] if regla != []]
            for regla in gramatica[A]:
                for i in range(len(regla)):
                    if regla[i] in nullable:
                        nueva_regla = regla[:i] + regla[i+1:]
                        if nueva_regla not in nuevas_reglas:
                            nuevas_reglas.append(nueva_regla)
                            
            gramatica[A] = nuevas_reglas
            for i in gramatica[A]:
                if i == []:
                    cambios = True
                    break
    return gramatica


def eliminar_inutiles(gramatica):
    """Elimina las producciones inútiles (no alcanzables o no generativas)."""
    generativas = set()
    alcanzables = set([list(gramatica.keys())[0]])  # Empezamos con el símbolo inicial

    # Encontramos las variables generativas
    while True:
        nueva_generativa = set(generativas)
        for A, reglas in gramatica.items():
            if A not in generativas:
                for regla in reglas:
                    if all(simbolo.islower() or simbolo in generativas for simbolo in regla):
                        nueva_generativa.add(A)
                        break
        if nueva_generativa == generativas:
            break
        generativas = nueva_generativa

    # Encontramos las variables alcanzables
    while True:
        nueva_alcanzable = set(alcanzables)
        for A in alcanzables:
            for regla in gramatica.get(A, []):
                for simbolo in regla:
                    if simbolo.isupper() and simbolo not in alcanzables:
                        nueva_alcanzable.add(simbolo)
        if nueva_alcanzable == alcanzables:
            break
        alcanzables = nueva_alcanzable

    # Eliminar producciones inútiles
    gramatica = {A: [regla for regla in reglas if all(simbolo in generativas and (simbolo in alcanzables or simbolo.islower()) for simbolo in regla)]
                 for A, reglas in gramatica.items() if A in generativas}

    return gramatica


def reemplazar_terminales(gramatica):
    """Reemplaza los terminales por nuevas variables (A -> a -> X)."""
    terminales_a_variables = {}
    nuevo_contador = 0

    for A, reglas in gramatica.items():
        for regla in reglas:
            for i in range(len(regla)):
                if isinstance(regla[i], str) and regla[i].islower():  # Si el símbolo es un terminal
                    if regla[i] not in terminales_a_variables:
                        nuevo_simbolo = f"X{nuevo_contador}"
                        terminales_a_variables[regla[i]] = nuevo_simbolo
                        gramatica[nuevo_simbolo] = [[regla[i]]]  # Crear una nueva producción para el terminal
                        nuevo_contador += 1
                    regla[i] = terminales_a_variables[regla[i]]  # Reemplazar el terminal por la nueva variable

    return gramatica


def reducir_terminales(gramatica):
    """Convierte producciones con más de dos símbolos en producciones válidas en CNF."""
    nuevas_producciones = {}
    nuevo_contador = 0

    # Recorremos todas las producciones
    for A, reglas in gramatica.items():
        nuevas_producciones[A] = []
        for regla in reglas:
            if len(regla) > 2:
                # Crear nuevas variables para producciones con más de dos símbolos
                nuevo_simbolo = f"X{nuevo_contador}"
                nuevo_contador += 1
                nuevas_producciones[nuevo_simbolo] = [regla[1:]]  # La parte de la regla a partir del segundo símbolo
                nuevas_producciones[A].append([regla[0], nuevo_simbolo])  # Se reemplaza por la nueva variable
            else:
                nuevas_producciones[A].append(regla)
    
    return nuevas_producciones


In [8]:
# Funcion principal para transformar una gramática CFG a CNF

def cfg_a_cnf(gramatica):
    """Convierte una gramática CFG a CNF aplicando los pasos necesarios."""
    # Paso 1: Eliminar producciones vacías (repetir hasta que no haya cambios)
    gramatica = eliminar_vacias(gramatica)
    
    # Paso 2: Eliminar producciones unitarias
    gramatica = eliminar_unitarias(gramatica)
    
    # Paso 3: Eliminar producciones inútiles
    gramatica = eliminar_inutiles(gramatica)
    
    # Paso 4: Reemplazar terminales por variables
    gramatica = reemplazar_terminales(gramatica)
    
    # Paso 5: Reducir producciones con más de dos símbolos
    gramatica = reducir_terminales(gramatica)
    
    return gramatica


In [9]:
def convertir_cnf(gramatica):

    simbolo_inicial = list(gramatica.keys())[0]

    # No terminales: claves del diccionario
    no_terminales = set(gramatica.keys())
    terminales = set()

    # Recorrer todas las reglas para encontrar terminales
    for reglas in gramatica.values():
        for produccion in reglas:
            for simbolo in produccion:
                if simbolo not in no_terminales and simbolo != 'epsilon':
                    terminales.add(simbolo)

    nueva_gramatica = {}
    contador = 0

    # Paso 1: Nuevo símbolo inicial si hace falta
    if any(simbolo_inicial in prod for reglas in gramatica.values() for prod in reglas):
        nueva_gramatica['S0'] = [[simbolo_inicial]]
        simbolo_inicial = 'S0'

    # Copiar las reglas originales
    for no_terminal, reglas in gramatica.items():
        nueva_gramatica[no_terminal] = reglas.copy()

    # Paso 2: Eliminar epsilon
    vacios = set()
    for nt, reglas in nueva_gramatica.items():
        for regla in reglas:
            if regla == ['epsilon']:
                vacios.add(nt)
    for nt in vacios:
        nueva_gramatica[nt] = [r for r in nueva_gramatica[nt] if r != ['epsilon']]
        for nt2, reglas in nueva_gramatica.items():
            nuevas = []
            for regla in reglas:
                if nt in regla:
                    nueva = [s for s in regla if s != nt]
                    if nueva and nueva != regla and nueva not in reglas:
                        nuevas.append(nueva)
            nueva_gramatica[nt2].extend(nuevas)

    # Paso 3: Eliminar producciones unitarias
    for nt in list(nueva_gramatica):
        nuevas = []
        for regla in nueva_gramatica[nt]:
            if len(regla) == 1 and regla[0] in nueva_gramatica and regla[0] != nt:
                nuevas.extend([r for r in nueva_gramatica[regla[0]] if r not in nueva_gramatica[nt]])
        nueva_gramatica[nt].extend(nuevas)  

    # Paso 4: Dividir producciones largas
    for nt in list(nueva_gramatica):
        nuevas = []
        for regla in nueva_gramatica[nt]:
            while len(regla) > 2:
                nuevo = f"X{contador}"
                contador += 1
                nueva_gramatica[nuevo] = [[regla[0], regla[1]]]
                regla = [nuevo] + regla[2:]
            nuevas.append(regla)
        nueva_gramatica[nt] = nuevas

    # Paso 5: Separar terminales si hay más de uno en la producción
    reemplazos = {}
    for nt in list(nueva_gramatica):
        for i, regla in enumerate(nueva_gramatica[nt]):
            if len(regla) == 2:
                for j in range(2):
                    simbolo = regla[j]
                    if simbolo in terminales:
                        if simbolo not in reemplazos:
                            nuevo = f"T{contador}"
                            contador += 1
                            nueva_gramatica[nuevo] = [[simbolo]]
                            reemplazos[simbolo] = nuevo
                        regla[j] = reemplazos[simbolo]

    return nueva_gramatica

In [11]:
# Ejemplo de uso de la transformación a CNF

Grammar_CFG = {
    'S': [['A', 'S', 'B']],
    'A': [['a', 'A', 'S'], ['a'], []],  
    'B': [['S', 'b', 'S'], ['A'], ['b', 'b']]
}

# Resultado esperado
# Grammar_CFG_CNF = {
#     'S': [['A', 'S'], ['P', 'B'], ['S', 'B']],
#     'A': [['R', 'S'], ['X', 'S'], ['a']],
#     'B': [['T', 'S'], ['Y', 'Y'], ['R', 'S'], ['X', 'S'], ['a']],
#     'X': [['a']],
#     'Y': [['b']],
#     'P': [['A', 'S']],
#     'R': [['X', 'A']],
#     'T': [['S', 'Y']]
# }


print("Grammar in CNF:", es_cnf(Grammar_CFG))
print("nueva_gramatica:", convertir_cnf(Grammar_CFG))

# print("Original Grammar:", Grammar_CFG, end="\n\n")
# print("Paso 1:", eliminar_vacias(Grammar_CFG), end="\n\n") # eliminar hasta que no queden
# print("Paso 2:", eliminar_unitarias(Grammar_CFG), end="\n\n")
# print("Paso 3:", eliminar_inutiles(Grammar_CFG), end="\n\n")
# # print("Paso 4:", reemplazar_terminales(Grammar_CFG), end="\n\n")
# print("Paso 5:", reducir_terminales(Grammar_CFG), end="\n\n")



Grammar in CNF: False
nueva_gramatica: {'S0': [['S'], ['X0', 'B']], 'S': [['X1', 'B']], 'A': [['X2', 'S'], ['a'], []], 'B': [['X3', 'S'], ['A'], ['T5', 'T5'], ['X4', 'S'], ['a'], []], 'X0': [['A', 'S']], 'X1': [['A', 'S']], 'X2': [['T6', 'A']], 'X3': [['S', 'T5']], 'X4': [['T6', 'A']], 'T5': [['b']], 'T6': [['a']]}


In [ ]:
Grammar_CNF = cfg_a_cnf(Grammar_CFG)
print("Transformed Grammar to CNF:", Grammar_CNF)

In [14]:
from collections import defaultdict

def cargar_gramatica2(ruta_archivo):
    reglas = {}
    casos_true = []
    casos_false = []
    seccion_actual = 'gramatica'  # 'gramatica', 'true', 'false'

    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        for linea in archivo:
            linea = linea.strip()
            
            # Saltar líneas vacías
            if not linea:
                continue
                
            # Saltar la línea "PCFG"
            if linea == 'PCFG':
                continue
                
            # Detectar secciones
            if linea.startswith('# True'):
                seccion_actual = 'true'
                continue
            elif linea.startswith('# False'):
                seccion_actual = 'false'
                continue
            
            # Procesar según la sección actual
            if seccion_actual == 'gramatica' and '->' in linea:
                # Parsear regla de gramática
                lhs, rhs = linea.split('->', 1)
                lhs = lhs.strip()
                alternativas = rhs.split('|')
                
                producciones = []
                for alt in alternativas:
                    alt = alt.strip()
                    # Extraer probabilidad entre corchetes
                    if '[' in alt and ']' in alt:
                        produccion_str = alt[:alt.rfind('[')].strip()
                        prob_str = alt[alt.rfind('[') + 1:alt.rfind(']')].strip()
                        try:
                            probabilidad = float(prob_str)
                        except ValueError:
                            probabilidad = 1.0
                    else:
                        produccion_str = alt
                        probabilidad = 1.0
                    
                    # Convertir producción a lista
                    if produccion_str:
                        # Dividir en símbolos individuales
                        produccion = []
                        i = 0
                        while i < len(produccion_str):
                            if produccion_str[i].isupper():
                                # No-terminal (una sola letra mayúscula)
                                produccion.append(produccion_str[i])
                                i += 1
                            else:
                                # Terminal (puede ser varias letras minúsculas)
                                j = i
                                while j < len(produccion_str) and produccion_str[j].islower():
                                    j += 1
                                if j > i:
                                    produccion.append(produccion_str[i:j])
                                    i = j
                                else:
                                    i += 1
                        
                        if produccion:
                            producciones.append((produccion, probabilidad))
                
                if producciones:
                    reglas[lhs] = producciones
                    
            elif seccion_actual == 'true':
                # Caso que debe ser aceptado
                if linea == 'ϵ':
                    casos_true.append([])  # Cadena vacía
                else:
                    # Dividir en símbolos individuales
                    simbolos = []
                    i = 0
                    while i < len(linea):
                        if linea[i].isupper():
                            simbolos.append(linea[i])
                            i += 1
                        else:
                            j = i
                            while j < len(linea) and linea[j].islower():
                                j += 1
                            if j > i:
                                simbolos.append(linea[i:j])
                                i = j
                            else:
                                i += 1
                    casos_true.append(simbolos)
                    
            elif seccion_actual == 'false':
                # Caso que debe ser rechazado
                if linea == 'ϵ':
                    casos_false.append([])  # Cadena vacía
                else:
                    # Dividir en símbolos individuales
                    simbolos = []
                    i = 0
                    while i < len(linea):
                        if linea[i].isupper():
                            simbolos.append(linea[i])
                            i += 1
                        else:
                            j = i
                            while j < len(linea) and linea[j].islower():
                                j += 1
                            if j > i:
                                simbolos.append(linea[i:j])
                                i = j
                            else:
                                i += 1
                    casos_false.append(simbolos)

    return reglas, casos_true, casos_false

In [15]:
gramatica_pcky, casos_true, casos_false = cargar_gramatica2('C:/Users/Claudia/Documents/GitHub/PAA_CKY/inputPCKY.txt')

print("Gramatica PCFG:", gramatica_pcky)
print("Casos True:", casos_true)
print("Casos False:", casos_false)

Gramatica PCFG: {'S': [(['A', 'B'], 0.9), (['B', 'C'], 0.1)], 'A': [(['B', 'A'], 0.5), (['a'], 0.5)], 'B': [(['C', 'C'], 0.7), (['b'], 0.3)], 'C': [(['A', 'B'], 0.6), (['a'], 0.4)]}
Casos True: [['ab'], ['bab'], ['aabab'], ['aaaaab'], ['abbaba'], ['bababa'], ['aaa'], ['bbaaa']]
Casos False: [[], ['a'], ['b'], ['ac'], ['bbb'], ['abb'], ['baa'], ['abba']]


In [19]:
# EXTENSION 2
# ampliar el algoritmo CKY a probabilıstico.

from collections import defaultdict

def cky_probabilistico(gramatica, palabra):
    
    # Convertir palabra a lista si es string
    if isinstance(palabra, str):
        palabra = list(palabra)
    
    # Caso especial: palabra vacía
    if not palabra:
        return 0.0
    
    n = len(palabra)
    
    # Tabla CKY: tabla[i][j] = dict {NoTerminal: prob_max}
    tabla = [[{} for _ in range(n)] for _ in range(n)]
    
    # Construir índices inversos de producciones
    inv_bin = defaultdict(list)  # (B,C) -> lista de (A, p)
    inv_uni = defaultdict(list)  # (a,) -> lista de (A, p)
    
    for A, producciones in gramatica.items():
        for rhs, p in producciones:
            # Convertir lista a tupla para usar como clave en diccionario
            rhs_tuple = tuple(rhs)
            if len(rhs) == 1:
                inv_uni[rhs_tuple].append((A, p))
            elif len(rhs) == 2:
                inv_bin[rhs_tuple].append((A, p))
            else:
                # CKY sólo funciona con gramáticas en CNF (prod. de 1 o 2 símbolos)
                raise ValueError(f"Producción {A} -> {rhs} no está en CNF")
    
    # Llenar diagonal (subcadenas de longitud 1)
    for i in range(n):
        terminal = palabra[i]
        for A, p in inv_uni.get((terminal,), []):
            # Maximizar probabilidad si ya existe entrada para A
            tabla[i][i][A] = max(tabla[i][i].get(A, 0), p)
    
    # Llenar resto de la tabla (subcadenas de longitud 2 a n)
    for longitud in range(2, n + 1):
        for i in range(n - longitud + 1):
            j = i + longitud - 1
            cell = tabla[i][j]
            
            # Probar todos los puntos de división posibles
            for k in range(i, j):
                left_cell = tabla[i][k]
                right_cell = tabla[k + 1][j]
                
                # Combinar todos los no-terminales de las subceldas
                for B, prob_B in left_cell.items():
                    for C, prob_C in right_cell.items():
                        # Buscar producciones A -> B C
                        for A, prob_A in inv_bin.get((B, C), []):
                            prob_total = prob_A * prob_B * prob_C
                            
                            # Actualizar si encontramos mejor probabilidad
                            if prob_total > cell.get(A, 0):
                                cell[A] = prob_total
    
    # Devolver probabilidad de 'S' (o 0 si no aparece)
    return tabla[0][n - 1].get('S', 0.0)


# Ejemplo de uso con tu formato
if __name__ == "__main__":
    # Tu gramática de ejemplo
    Grammar2_pcfg = {
        'S': [
            (['A','B'], 0.4),      # S → A B (prob 0.4)
            (['C','D'], 0.3),      # S → C D (prob 0.3)
            (['C','B'], 0.2),      # S → C B (prob 0.2)
            (['S','S'], 0.1)       # S → S S (prob 0.1)
        ],
        'A': [
            (['B','C'], 0.7),      # A → B C (prob 0.7)
            (['a'], 0.3)           # A → a (prob 0.3)
        ],
        'B': [
            (['S','C'], 0.5),      # B → S C (prob 0.5)
            (['b'], 0.5)           # B → b (prob 0.5)
        ],
        'C': [
            (['D','D'], 0.8),      # C → D D (prob 0.8)
            (['b'], 0.2)           # C → b (prob 0.2)
        ],
        'D': [
            (['B','A'], 1.0)       # D → B A (prob 1.0)
        ]
    }
    
    # Probar diferentes oraciones
    oraciones = [
        ['a', 'b'],           # Simple: A(a) B(b) -> S  
        ['b', 'b'],           # C(b) B(b) -> S
        ['a', 'b', 'a', 'b'], # Más compleja
        ['b', 'a', 'b']       # Otra combinación
    ]
    
    for palabra in oraciones:
        prob = cky_probabilistico(Grammar2_pcfg, palabra)
        print(f"'{' '.join(palabra)}': probabilidad = {prob:.9f}")
    
    # Análisis detallado de 'a b'
    print("\n--- Análisis de 'a b' ---")
    print("Posibles derivaciones:")
    print("1. S → A B")
    print("   - A → a (prob 0.3)")  
    print("   - B → b (prob 0.5)")
    print("   - Total: 0.4 × 0.3 × 0.5 = 0.06")
    print()
    print("2. S → C B") 
    print("   - C → b (prob 0.2) - NO FUNCIONA, necesitamos 'a' no 'b'")
    print("   - Esta derivación no es posible")
    print()
    print("Resultado esperado: 0.06")

'a b': probabilidad = 0.060000000
'b b': probabilidad = 0.020000000
'a b a b': probabilidad = 0.000360000
'b a b': probabilidad = 0.000000000

--- Análisis de 'a b' ---
Posibles derivaciones:
1. S → A B
   - A → a (prob 0.3)
   - B → b (prob 0.5)
   - Total: 0.4 × 0.3 × 0.5 = 0.06

2. S → C B
   - C → b (prob 0.2) - NO FUNCIONA, necesitamos 'a' no 'b'
   - Esta derivación no es posible

Resultado esperado: 0.06


In [18]:
for palabra in casos_true:
    prob = cky_probabilistico(gramatica_pcky, palabra)
    print(f"Palabra True: {' '.join(palabra)} - Probabilidad: {prob:.6f}")

print

for palabra in casos_false:
    prob = cky_probabilistico(gramatica_pcky, palabra)
    print(f"Palabra False: {' '.join(palabra)} - Probabilidad: {prob:.6f}")

Palabra True: ab - Probabilidad: 0.000000
Palabra True: bab - Probabilidad: 0.000000
Palabra True: aabab - Probabilidad: 0.000000
Palabra True: aaaaab - Probabilidad: 0.000000
Palabra True: abbaba - Probabilidad: 0.000000
Palabra True: bababa - Probabilidad: 0.000000
Palabra True: aaa - Probabilidad: 0.000000
Palabra True: bbaaa - Probabilidad: 0.000000
Palabra False:  - Probabilidad: 0.000000
Palabra False: a - Probabilidad: 0.000000
Palabra False: b - Probabilidad: 0.000000
Palabra False: ac - Probabilidad: 0.000000
Palabra False: bbb - Probabilidad: 0.000000
Palabra False: abb - Probabilidad: 0.000000
Palabra False: baa - Probabilidad: 0.000000
Palabra False: abba - Probabilidad: 0.000000
